Taking the plots out of the file ev_experiments to safe the data 

In [8]:
from __future__ import annotations

import os
from typing import Callable, Dict, Optional, Tuple, List

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed

from ev_core import (
    EVStagHuntModel,
    set_initial_adopters,
    final_mean_adoption_vs_ratio,
    phase_sweep_X0_vs_ratio,
)
from ev_plotting import (
    plot_fanchart,
    plot_spaghetti,
    plot_density,
    plot_ratio_sweep,
    plot_phase_plot,
    
)

from ev_experiments import (
    plot_intervention_fanchart,
    phase_sweep_df,
    ratio_sweep_df
    

)

Pickle 

In [ ]:
#first code from second brain 

import os
import glob
import pickle
import pandas as pd


def load_all_pickled_dataframes(base_dir="."):
    """
    Scan for plots_{label} folders, open all *_data.pkl files,
    and collect all DataFrames inside them.

    Returns a nested dict:
        dfs[label][dataset_name][key] = DataFrame

    where:
        label         = scenario label, e.g. "InitialAdoption0.3"
        dataset_name  = "intervention", "spaghetti_density", "phase", "ratio_sweep", ...
        key           = the key inside the saved dict (e.g. 'baseline_df', 'subsidy_df', 'phase_df', 'sweep_df')
    """
    dfs = {}

    # Find all folders like plots_InitialAdoption0.3, plots_InitialAdoption0.5, ...
    for folder in glob.glob(os.path.join(base_dir, "plots_*")):
        label = os.path.basename(folder).replace("plots_", "")
        dfs[label] = {}

        # Find all *_data.pkl inside this folder
        for pkl_path in glob.glob(os.path.join(folder, "*_data.pkl")):
            dataset_name = os.path.basename(pkl_path).replace("_data.pkl", "")

            with open(pkl_path, "rb") as f:
                obj = pickle.load(f)

            # If the pickle is a dict (as created by save_scenario_data), scan its values
            if isinstance(obj, dict):
                df_dict = {}
                for key, value in obj.items():
                    if isinstance(value, pd.DataFrame):
                        df_dict[key] = value

                # Only store if we actually found some DataFrames in this pickle
                if df_dict:
                    dfs[label][dataset_name] = df_dict

    return dfs


if __name__ == "__main__":
    all_dfs = load_all_pickled_dataframes()

    # Example: access some specific DataFrames
    # all_dfs["InitialAdoption0.3"]["intervention"]["baseline_df"]
    # all_dfs["InitialAdoption0.3"]["phase"]["phase_df"]
    # all_dfs["InitialAdoption0.3"]["ratio_sweep"]["sweep_df"]

    # Quick sanity print
    for label, datasets in all_dfs.items():
        print(f"\n=== {label} ===")
        for name, df_dict in datasets.items():
            for key, df in df_dict.items():
                print(f"{name}.{key}: shape={df.shape}")


In [ ]:
#got error here
all_dfs["InitialAdoption0.3"]["intervention"]["baseline_df"]
all_dfs["InitialAdoption0.3"]["intervention"]["subsidy_df"]
all_dfs["InitialAdoption0.3"]["spaghetti_density"]["traces_df"]
all_dfs["InitialAdoption0.3"]["phase"]["phase_df"]
all_dfs["InitialAdoption0.3"]["ratio_sweep"]["sweep_df"]


KeyError: 'data_InitialAdoption0.3'

In [ ]:
#fixed code from second brain, this do work

import os
import glob
import pickle
import pandas as pd
import numpy as np
import types
import traceback

def find_pkl_files(base_dir="."):
    """Return list of (label, pkl_path) for plots_{label}/*_data.pkl"""
    files = []
    for folder in glob.glob(os.path.join(base_dir, "plots_*")):
        label = os.path.basename(folder).replace("plots_", "")
        for pkl_path in glob.glob(os.path.join(folder, "*_data.pkl")):
            files.append((label, pkl_path))
    return files

def try_load_pickle(path):
    """
    Try to load a pickle with a couple of fallbacks.
    Returns (obj, error) where error is None on success or an exception string.
    """
    errors = []
    for enc in (None, "latin1"):
        try:
            with open(path, "rb") as f:
                if enc is None:
                    obj = pickle.load(f)
                else:
                    # pickle.load accepts encoding param in Python3
                    obj = pickle.load(f, encoding=enc)
            return obj, None
        except Exception as e:
            errors.append(f"enc={enc!r}: {repr(e)}")
    return None, " | ".join(errors)

def is_dataframe_like(obj):
    return isinstance(obj, pd.DataFrame)

def is_series_like(obj):
    return isinstance(obj, pd.Series)

def is_numpy_structured_array(obj):
    return isinstance(obj, np.ndarray) and obj.dtype.names is not None

def try_convert_to_dataframe(obj):
    """
    Attempt safe conversions to DataFrame:
      - pandas.Series -> to_frame()
      - objects with to_pandas() or to_dataframe() method
      - numpy structured arrays -> DataFrame(...)
      - other np.ndarray / list-like -> DataFrame(...) if succeeds
    Returns (df, error)
    """
    try:
        if is_dataframe_like(obj):
            return obj, None
        if is_series_like(obj):
            return obj.to_frame(), None
        # common custom table-like: pyarrow.Table, xarray, etc.
        if hasattr(obj, "to_pandas") and callable(obj.to_pandas):
            try:
                return obj.to_pandas(), None
            except Exception as e:
                # continue to other attempts
                pass
        if hasattr(obj, "to_dataframe") and callable(obj.to_dataframe):
            try:
                return obj.to_dataframe(), None
            except Exception:
                pass
        if is_numpy_structured_array(obj):
            return pd.DataFrame(obj), None
        # try generic conversion (may raise)
        if isinstance(obj, (list, tuple, np.ndarray, dict)):
            df = pd.DataFrame(obj)
            return df, None
    except Exception as e:
        return None, repr(e)
    return None, "not-convertible"

def _repr_type(obj):
    try:
        return type(obj).__name__
    except Exception:
        return str(type(obj))

def recursively_find_dataframes(obj, path="root", results=None, seen=None):
    """
    Walk object recursively and collect DataFrames.
    results: list of dicts -> appended entries: {'path': path, 'df': DataFrame, 'type': type_name}
    seen: set of object ids to avoid infinite recursion
    """
    if results is None:
        results = []
    if seen is None:
        seen = set()

    oid = id(obj)
    if oid in seen:
        return results
    seen.add(oid)

    # Direct DataFrame
    if is_dataframe_like(obj):
        results.append({"path": path, "df": obj, "type": _repr_type(obj)})
        return results

    # Series -> convert
    if is_series_like(obj):
        try:
            df = obj.to_frame()
            results.append({"path": path, "df": df, "type": _repr_type(obj) + " (Series->DataFrame)"})
        except Exception:
            pass
        return results

    # convertible objects
    conv, conv_err = try_convert_to_dataframe(obj)
    if conv is not None:
        results.append({"path": path, "df": conv, "type": _repr_type(obj) + " (converted)"})
        return results

    # dict-like
    if isinstance(obj, dict):
        for k, v in obj.items():
            child_path = f"{path}['{k}']"
            recursively_find_dataframes(v, child_path, results, seen)
        return results

    # list/tuple/set
    if isinstance(obj, (list, tuple, set)):
        for i, item in enumerate(obj):
            child_path = f"{path}[{i}]"
            recursively_find_dataframes(item, child_path, results, seen)
        return results

    # objects with __dict__ (custom objects)
    if hasattr(obj, "__dict__"):
        try:
            for k, v in vars(obj).items():
                child_path = f"{path}.{k}"
                recursively_find_dataframes(v, child_path, results, seen)
            return results
        except Exception:
            pass

    # fallback: nothing found
    return results

def load_all_pickled_dataframes(base_dir="."):
    """
    Returns structure:
      {
        label: {
          dataset_name: [
             {"path": path_in_obj, "df": DataFrame, "type": type_name, "orig_file": pkl_path}
          ],
          ...
        },
        ...
      }
    And prints a summary of what it found and what failed.
    """
    found = {}
    failed_loads = []
    no_dfs = []

    files = find_pkl_files(base_dir)
    if not files:
        print("No files found matching plots_*/ *_data.pkl under", os.path.abspath(base_dir))
        return found

    for label, pkl_path in files:
        dataset_name = os.path.basename(pkl_path).replace("_data.pkl", "")
        found.setdefault(label, {})
        obj, err = try_load_pickle(pkl_path)
        if err:
            failed_loads.append({"label": label, "file": pkl_path, "error": err})
            continue

        # recursively search
        results = recursively_find_dataframes(obj, path="root")
        if not results:
            no_dfs.append({"label": label, "file": pkl_path, "note": "no DataFrames found; top-type=" + _repr_type(obj)})
            continue

        # annotate with orig file and add to map
        annotated = []
        for r in results:
            r2 = dict(r)  # copy
            r2["orig_file"] = pkl_path
            # optionally store shape and columns snapshot to inspect quickly
            try:
                r2["shape"] = r2["df"].shape
                # small preview of columns (limit length)
                r2["cols"] = list(r2["df"].columns[:10]) if hasattr(r2["df"], "columns") else []
            except Exception:
                r2["shape"] = None
                r2["cols"] = []
            annotated.append(r2)
        found[label].setdefault(dataset_name, []).extend(annotated)

    # Print diagnostics
    print("\n=== Summary ===")
    total_dfs = sum(len(v2) for label_map in found.values() for v2 in label_map.values())
    print(f"Pickles scanned: {len(files)}    DataFrames found: {total_dfs}\n")

    if found:
        print("Found DataFrames (sample):")
        for label, datasets in found.items():
            print(f"  label={label}")
            for dsname, items in datasets.items():
                for item in items:
                    print(f"    - {dsname} @ {item['path']}  file={os.path.basename(item['orig_file'])}  shape={item.get('shape')}  type={item.get('type')}")
    else:
        print("No DataFrames discovered inside any pickle.")

    if failed_loads:
        print("\nFailed to load the following pickles:")
        for f in failed_loads:
            print(f"  - {os.path.basename(f['file'])} (label={f['label']}): {f['error']}")

    if no_dfs:
        print("\nPickles loaded successfully but contained no DataFrames:")
        for n in no_dfs:
            print(f"  - {os.path.basename(n['file'])} (label={n['label']}): {n['note']}")

    return found

if __name__ == "__main__":
    all_dfs = load_all_pickled_dataframes(base_dir=".")
    # Example how to access:
    # all_dfs["InitialAdoption0.3"]["intervention"][0]["df"]  -> actual DataFrame
    # Print quick shapes:
    print("\n=== Quick shapes list ===")
    for label, datasets in all_dfs.items():
        for dsname, items in datasets.items():
            for it in items:
                print(f"{label} / {dsname} / {it['path']} -> {it.get('shape')}")


No files found matching plots_*/ *_data.pkl under c:\Users\tjk20\Documents\master_csp\model_based\Model-Based-Decisions-Code\Assignment_3_EV_Stag_Hunt_Game\Assignment3

=== Quick shapes list ===


Fanchart

In [5]:
def plot_fanchart(traces_df: pd.DataFrame, out_path: Optional[str] = None) -> str:
    """Plot fan charts (quantile bands) for baseline vs subsidy using traces DF.

    traces_df columns: ['group', 'trial', 'time', 'X'] where group in {'baseline','subsidy'}.
    """
    if traces_df.empty:
        raise ValueError("traces_df is empty")

    groups = ["baseline", "subsidy"]
    fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)

    for j, group in enumerate(groups):
        gdf = traces_df[traces_df["group"] == group]

        # Compute quantiles by time across trials
        q = gdf.groupby("time")["X"].quantile([0.10, 0.25, 0.75, 0.90]).unstack(level=1)
        mean = gdf.groupby("time")["X"].mean()
        t = mean.index.to_numpy()

        ax = axes[0, j]
        ax.fill_between(t, q[0.10], q[0.90], color=("steelblue" if group == "baseline" else "darkorange"), alpha=0.15, label="10–90%")
        ax.fill_between(t, q[0.25], q[0.75], color=("steelblue" if group == "baseline" else "darkorange"), alpha=0.30, label="25–75%")

        # Overlay some traces for context (sample up to 100 trials)
        trial_ids = gdf["trial"].unique()
        rng = np.random.default_rng(123)
        sample = rng.choice(trial_ids, size=min(100, len(trial_ids)), replace=False)
        for tr in sample:
            tr_df = gdf[gdf["trial"] == tr]
            ax.plot(tr_df["time"], tr_df["X"], color=("steelblue" if group == "baseline" else "darkorange"), alpha=0.1, linewidth=0.8)

        ax.plot(t, mean, color=("steelblue" if group == "baseline" else "darkorange"), linewidth=2, label="mean")
        ax.set_title(f"{group.capitalize()} adoption")
        ax.set_xlabel("Time")
        ax.set_ylabel("X(t)")
        ax.set_ylim(0, 1)
        ax.legend(loc="lower right")

        # Final X(T) histogram
        t_max = int(gdf["time"].max())
        final_vals = gdf[gdf["time"] == t_max].groupby("trial")["X"].mean().to_numpy()
        axes[1, j].hist(final_vals, bins=20, color=("steelblue" if group == "baseline" else "darkorange"), alpha=0.8)
        axes[1, j].set_title(f"{group.capitalize()} final X(T)")
        axes[1, j].set_xlabel("X(T)")
        axes[1, j].set_ylabel("Count")

    if out_path is None:
        out_path = _default_plot_path("ev_intervention_fanchart.png")
    fig.savefig(out_path, dpi=140)
    plt.close(fig)
    return out_path

In [11]:
fanchart_path = plot_intervention_fanchart(
    baseline_X,
    subsidy_X,
    out_path=f"plots_{label}/fanchart_{label}.png", 
)
print("Saved fanchart image:", fanchart_path)
        

NameError: name 'baseline_X' is not defined

Phase 

In [9]:
def plot_phase_plot(phase_df: pd.DataFrame, out_path: Optional[str] = None) -> str:
    """Plot heatmap from tidy DataFrame with columns ['X0','ratio','X_final']."""
    # Pivot to matrix for imshow
    pivot = phase_df.pivot(index="ratio", columns="X0", values="X_final").sort_index().sort_index(axis=1)
    ratios = pivot.index.to_numpy()
    X0s = pivot.columns.to_numpy()

    plt.figure(figsize=(7, 4))
    im = plt.imshow(
        pivot.to_numpy(),
        origin="lower",
        extent=[X0s[0], X0s[-1], ratios[0], ratios[-1]],
        aspect="auto",
        vmin=0.0,
        vmax=1.0,
        cmap="plasma",
    )
    plt.colorbar(im, label="Final adopters X*")
    plt.xlabel("X0 (initial adoption)")
    plt.ylabel("a_I / b (initial payoff ratio)")
    plt.title("Network phase plot: X* over X0 and a_I/b")

 # Overlay threshold X = 1/ratio
    X_thresh = 1.0 / ratios
    X_thresh_clipped = np.clip(X_thresh, 0.0, 1.0)
    plt.plot(X_thresh_clipped, ratios, color="white", linestyle="--", linewidth=1.5, label="X = b / a_I (initial)")
    plt.legend(loc="upper right")

    if out_path is None:
        out_path = _default_plot_path("ev_phase_plot.png")
    plt.savefig(out_path, dpi=140, bbox_inches="tight")
    plt.close()
    return out_path

In [10]:
# Also run the phase plot of X* over (X0, a_I/b) and save it
phase_df = phase_sweep_df(
    max_workers=max_workers,
    backend="thread",
    X0_values=np.linspace(0.0, 1.0, 21),
    ratio_values=np.linspace(0.8, 3.5, 31),
    batch_size=8,
    T=200,
    strategy_choice_func=strategy_choice_func,
    tau=tau,
    scenario_kwargs=scenario
    )
phase_path = plot_phase_plot(phase_df, out_path=f"plots_{label}/phase_{label}.png")
print("Saved phase plot:", phase_path)

NameError: name 'max_workers' is not defined

Spaghetti 

In [ ]:
def plot_spaghetti(traces_df: pd.DataFrame, *, max_traces: int = 100, alpha: float = 0.15, out_path: Optional[str] = None) -> str:
    """Spaghetti plot from traces DF for baseline vs subsidy."""
    groups = ["baseline", "subsidy"]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
    rng = np.random.default_rng(123)

    for j, group in enumerate(groups):
        gdf = traces_df[traces_df["group"] == group]
        trial_ids = gdf["trial"].unique()
        sample = rng.choice(trial_ids, size=min(max_traces, len(trial_ids)), replace=False)
        ax = axes[j]
        for tr in sample:
            tr_df = gdf[gdf["trial"] == tr]
            ax.plot(tr_df["time"], tr_df["X"], color=("steelblue" if group == "baseline" else "darkorange"), alpha=alpha, linewidth=0.8)
        ax.set_title(f"{group.capitalize()} traces")
        ax.set_xlabel("Time")
        ax.set_ylabel("X(t)")
        ax.set_ylim(0, 1)

    if out_path is None:
        out_path = _default_plot_path("ev_spaghetti.png")
    fig.savefig(out_path, dpi=140)
    plt.close(fig)
    return out_path



In [ ]:
def plot_density(traces_df: pd.DataFrame, *, x_bins: int = 50, time_bins: Optional[int] = None, out_path: Optional[str] = None) -> str:
    """Time-evolving density plot (2D histogram) from traces DF."""
    groups = ["baseline", "subsidy"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)

    for j, group in enumerate(groups):
        gdf = traces_df[traces_df["group"] == group]
        T = int(gdf["time"].max()) + 1
        if time_bins is None:
            bins_time = T
        else:
            bins_time = time_bins
        hb = axes[j].hist2d(gdf["time"].to_numpy(), gdf["X"].to_numpy(), bins=[bins_time, x_bins], range=[[0, T - 1], [0.0, 1.0]], cmap="magma")
        axes[j].set_title(f"{group.capitalize()} density: time vs X(t)")
        axes[j].set_xlabel("Time")
        axes[j].set_ylabel("X(t)")
        fig.colorbar(hb[3], ax=axes[j], label="count")

    if out_path is None:
        out_path = _default_plot_path("ev_density.png")
    fig.savefig(out_path, dpi=140)
    plt.close(fig)
    return out_path

In [ ]:
# Spaghetti and time-evolving density plots
        # Use a larger trial count for clearer trace/density visuals
n_trials_spaghetti = 100
T_spaghetti = 200

baseline_X, baseline_I, subsidy_X, subsidy_I, baseline_df2, subsidy_df2 = collect_intervention_trials(
    n_trials=n_trials_spaghetti,
    T=T_spaghetti,
    scenario_kwargs=scenario,
    subsidy_params=subsidy,
    max_workers=max_workers,
    seed_base=seed_base,
    strategy_choice_func=strategy_choice_func,
    tau=tau,
)
traces_df = traces_to_long_df(baseline_X, subsidy_X)
spaghetti_path = plot_spaghetti(traces_df, max_traces=100, alpha=0.15, out_path=f"plots_{label}/spaghetti_{label}.png")
        
print("Saved spaghetti plot:", spaghetti_path)

density_path = plot_density(traces_df, x_bins=50, time_bins=T_spaghetti, out_path=f"plots_{label}/density_{label}.png")
print("Saved time-evolving density plot:", density_path)


Ratio Sweep 

In [ ]:
def plot_ratio_sweep(sweep_df: pd.DataFrame, out_path: Optional[str] = None) -> str:
    """Plot X* vs ratio from a DataFrame with columns ['ratio','X_mean']."""
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(sweep_df["ratio"], sweep_df["X_mean"], color="C0", lw=2)
    ax.set_xlabel("a_I / b (ratio)")
    ax.set_ylabel("Final adoption X*")
    ax.set_title("X* vs ratio")
    ax.set_ylim(0.0, 1.0)
    ax.grid(True, alpha=0.25)
    if out_path is None:
        out_path = _default_plot_path("ev_ratio_sweep.png")
    fig.savefig(out_path, dpi=140, bbox_inches="tight")
    plt.close(fig)
    return out_path

In [ ]:
sweep_df = ratio_sweep_df( 
    X0_frac=scenario.get("X0_frac", 0.40),
    ratio_values=np.linspace(0.8, 3.5, 31),
    scenario_kwargs=scenario,
    T=200,
    batch_size=8,
    strategy_choice_func=strategy_choice_func,
    tau=tau
)
sweep_path = plot_ratio_sweep(sweep_df, out_path=f"plots_{label}/ratio_sweep_{label}.png")
print("Saved ratio sweep plot:", sweep_path)